In [ ]:
from huggingface_hub import login

login(token="")

In [ ]:
# gemma_classifier_multi_topic.py
# Multi-topic classifier for 10 policy topics in Danish articles

import pandas as pd
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import re


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_CSV = Path("/work/pilot/data/clean/chunk_993_1322.csv")
OUTPUT_CSV = Path("/work/pilot/data/processed/993_1322_results.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/gemma-2-9b-it"
MAX_ARTICLE_LENGTH = 2000


# ============================================================
# LOAD MODEL
# ============================================================

print(f"Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if tokenizer.chat_template is None:
    tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'user' %}{{ '<start_of_turn>user\n' + message['content'] + '<end_of_turn>\n' }}{% elif message['role'] == 'assistant' %}{{ '<start_of_turn>model\n' + message['content'] + '<end_of_turn>\n' }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<start_of_turn>model\n' }}{% endif %}"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    torch_dtype="auto", 
    device_map="auto"
)

print(f"Model loaded on {model.device}")


# ============================================================
# PROMPT TEMPLATE
# ============================================================

def create_prompt(article_text):
    """Creating classification prompt"""
    
    prompt = f"""Du er en præcis klassifikationsekspert. Din opgave er at identificere specifikke politiske emner i danske artikler.

VIGTIG REGEL: Svar kun JA hvis artiklen DIREKTE diskuterer det specifikke emne. Vær STRENG - hvis du er i tvivl, svar NEJ.

EMNER (læs hver definition nøje):

1. PRIVATISERING: Artiklen diskuterer overførsel af offentlige opgaver eller velfærdsydelser til private aktører, eller debatter om privat versus offentlig drift af serviceområder
   → JA eksempel: "Kommunen overvejer at udlicitere hjemmeplejen til private"
   → NEJ eksempel: "Private virksomheder åbner i byen"

2. BESPARELSER OG EFFEKTIVITET: Artiklen diskuterer at reducere offentlige udgifter, forbedre effektivitet, eller omorganisere kommunale services for at spare penge uden at forringe velfærdskvaliteten
   → JA eksempel: "Kommunen kan spare ved at omorganisere uden at påvirke servicen"
   → NEJ eksempel: "Budgettet diskuteres"

3. KOMMUNAL SKATTEPOLITIK: Artiklen diskuterer kommunale skattesatser, lokale skatteniveauer, eller forslag om at hæve eller sænke skatten på kommunalt niveau (fx i Aarhus eller andre kommuner)
   → JA eksempel: "Kommuneskatten skal sættes op/ned"
   → NEJ eksempel: "Skatteindtægter stiger"

4. LOKALE ERHVERVSVILKÅR: Artiklen diskuterer kommunale tiltag eller beslutninger der påvirker hvor let eller dyrt det er at drive virksomhed lokalt (fx gebyrer, tilladelser, infrastrukturomkostninger)
   → JA eksempel: "Kommunen foreslår at sænke erhvervsafgifter for at tiltrække firmaer"
   → NEJ eksempel: "Nye virksomheder åbner"

5. SKOLEBLANDING OG OPTAG: Artiklen diskuterer politikker om at blande elever efter baggrund eller regulere skolevalg, især hvis det berører lighed, segregering, eller frit valg
   → JA eksempel: "Politikere vil blande elever på tværs af social baggrund"
   → NEJ eksempel: "Skoler får flere elever"

6. MOSKÉER OG RELIGIØSE BYGNINGER: Artiklen diskuterer at tillade eller blokere opførelse af moskéer eller andre religiøse bygninger, eller bredere debatter om deres rolle i lokalsamfundet
   → JA eksempel: "Politikere debatterer begrænsninger på moskébyggeri"
   → NEJ eksempel: "Religiøse bygninger i byen"

7. PARKERINGSPOLITIK: Artiklen diskuterer at etablere, fjerne eller regulere parkeringspladser, eller debatter om hvordan byrum skal bruges til biler versus andre formål
   → JA eksempel: "Byrådet diskuterer at fjerne parkeringspladser til fordel for cykelstier"
   → NEJ eksempel: "Folk parkerer i byen"

8. HAVN- OG HAVNEUDVIKLING: Artiklen diskuterer fremtiden for havneområder (fx omdannelse, byggeforbud, ophævelse af restriktioner) og hvordan de skal bruges i byplanlægningen
   → JA eksempel: "Havnestoppet skal gøres permanent eller ophæves"
   → NEJ eksempel: "Havnen er travl med containere"

9. BILTRAFIK OG BYMIDTADGANG: Artiklen diskuterer politikker for at begrænse eller regulere biltrafik i centrale byområder (fartgrænser, bilfrie zoner, emissionsregler)
   → JA eksempel: "Nye tiltag skal reducere biltrafikken i midtbyen med ensretninger"
   → NEJ eksempel: "Meget trafik i byen"

10. KONGELUNDEN ELLER STORE STADIONPROJEKTER: Artiklen diskuterer Kongelunden-stadionet eller lignende store kommunale sports- eller kulturprojekter - om de skal fortsætte, ændres, eller stoppes
    → JA eksempel: "Omkostninger til det nye stadion i Kongelunden diskuteres politisk"
    → NEJ eksempel: "AGF spiller kamp"

KLASSIFIKATION STEP-BY-STEP:
1. Læs artiklen grundigt
2. For hvert emne: Er det SPECIFIKKE emne direkte diskuteret?
3. Hvis artiklen kun tangerer emnet eller bruger lignende ord uden at diskutere det specifikke indhold → NEJ
4. Hvis i tvivl → NEJ (vær streng!)

SVARFORMAT: 
Præcis 10 svar adskilt af komma og mellemrum.
Format: NEJ, NEJ, JA, NEJ, NEJ, NEJ, NEJ, NEJ, NEJ, NEJ

ARTIKEL:
{article_text}

Klassificer nu artiklen for alle 10 emner. Svar KUN med sekvensen:"""
    
    return prompt


# ============================================================
# CLASSIFICATION
# ============================================================

def classify_article(article_text, article_id):
    """Classifying article"""
    
    if len(article_text) > MAX_ARTICLE_LENGTH:
        article_text = article_text[:MAX_ARTICLE_LENGTH]
    
    prompt = create_prompt(article_text)
    messages = [{"role": "user", "content": prompt}]
    
    text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(
        text=[text],
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, outputs)
    ]
    
    output_text = tokenizer.batch_decode(
        generated_ids_trimmed, 
        skip_special_tokens=True, 
        clean_up_tokenization_spaces=False
    )
    
    result = output_text[0].strip()
    labels = parse_sequence(result)
    
    return {
        "labels": labels,
        "model_output": result
    }


def parse_sequence(model_output):
    """Parsing model output"""
    
    output = model_output.upper().strip()
    parts = re.split(r'[,\s]+', output)
    
    labels = []
    for part in parts:
        part_clean = part.strip()
        if 'JA' in part_clean and 'NEJ' not in part_clean:
            labels.append(1)
        elif 'NEJ' in part_clean:
            labels.append(0)
    
    if len(labels) < 10:
        labels.extend([0] * (10 - len(labels)))
    elif len(labels) > 10:
        labels = labels[:10]
    
    return labels


# ============================================================
# LOAD DATA
# ============================================================

print("Loading data...")
df = pd.read_csv(INPUT_CSV, dtype=str)

TEXT_COL = next(
    (c for c in df.columns if c.lower() in ["text", "content", "raw_text", "article_text"]), 
    df.columns[2] if len(df.columns) > 2 else df.columns[1]
)

print(f"Text column: {TEXT_COL}")
print(f"Total articles: {len(df)}\n")


# ============================================================
# PROCESS ARTICLES
# ============================================================

print("Starting classification...\n")

data = []
start_time = time.time()

for idx, row in df.iterrows():
    identifier = row.get("id", row.get("file", idx))
    article_text = str(row.get(TEXT_COL, "")).strip()
    
    if not article_text:
        data.append({
            "id": identifier,
            "url": row.get("url", ""),
            "text_excerpt": "",
            "topic_1": 0, "topic_2": 0, "topic_3": 0, "topic_4": 0, "topic_5": 0,
            "topic_6": 0, "topic_7": 0, "topic_8": 0, "topic_9": 0, "topic_10": 0,
            "model_output": "Empty article"
        })
        continue
    
    try:
        classification = classify_article(article_text, identifier)
        labels = classification['labels']
        
        data.append({
            "id": identifier,
            "url": row.get("url", ""),
            "text_excerpt": article_text[:300].replace("\n", " "),
            "topic_1": labels[0], "topic_2": labels[1], "topic_3": labels[2],
            "topic_4": labels[3], "topic_5": labels[4], "topic_6": labels[5],
            "topic_7": labels[6], "topic_8": labels[7], "topic_9": labels[8],
            "topic_10": labels[9],
            "model_output": classification["model_output"]
        })
        
    except Exception as e:
        data.append({
            "id": identifier,
            "url": row.get("url", ""),
            "text_excerpt": article_text[:300].replace("\n", " "),
            "topic_1": 0, "topic_2": 0, "topic_3": 0, "topic_4": 0, "topic_5": 0,
            "topic_6": 0, "topic_7": 0, "topic_8": 0, "topic_9": 0, "topic_10": 0,
            "model_output": f"Error: {str(e)}"
        })
    
    # Progress update every 50 articles
    if (idx + 1) % 50 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / (idx + 1)
        remaining = (len(df) - (idx + 1)) * avg_time
        print(f"Progress: {idx + 1}/{len(df)} | ETA: {remaining/60:.1f} min")
    
    # Autosave every 100 articles
    if (idx + 1) % 100 == 0:
        temp_df = pd.DataFrame(data)
        temp_df.to_csv(OUTPUT_CSV, index=False)


# ============================================================
# SAVE RESULTS
# ============================================================

df_results = pd.DataFrame(data)
df_results.to_csv(OUTPUT_CSV, index=False)

total_time = time.time() - start_time
topic_counts = {i+1: sum(d[f'topic_{i+1}'] for d in data) for i in range(10)}

print(f"\nClassification complete")
print(f"Output: {OUTPUT_CSV}")
print(f"Total articles: {len(data)}")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Average per article: {total_time/len(data):.1f} seconds\n")

print("Topic counts:")
for topic_num, count in topic_counts.items():
    print(f"  Topic {topic_num}: {count} ({count/len(data)*100:.1f}%)")
